<a href="https://colab.research.google.com/github/linlingfa/Geospatial_Practice/blob/main/ASSIP_Prescreening_Lin_Aaron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Link to GitHub Public Repository: https://github.com/linlingfa/Geospatial_Practice

# ASSIP 2025 — Pre-Screening Assignment

Make a copy of this notebook and rename it `ASSIP_Prescreening_LastName_FirstName`.
Share it with me by **Monday noon** via email at dbor[at]gmu.edu,
or add borkiproino[at]gmail.com as a viewer.

---

### AI Policy
Permitted. If used, state the tool, paste your prompt, and explain the output in your own words.
Show your thinking.

### Markdown
Use markdown cells to tell a story. Before each code block, explain what you are doing and why.
After, interpret what you see.

### Data
All files are here: https://drive.google.com/drive/folders/1kCa3dDEiAJzfMpux_-Wyu9FO3Ub5ukFL?usp=sharing

---

### Quiz 3 — Git Commits
Push your completed notebook to a public GitHub repository.
Make at least 5 meaningful commits as you work - not one single commit at the end.
Each commit message should describe what you did and why.
Share your github repository link.

### Quiz 4 - Video Pitch
Record a screen-share video (max 6 minutes) walking through your notebook.
Cover what you did in each quiz, what you found, and what surprised you.
If you used AI, show your prompts and explain what you changed.
Attach the video in your final submission (via email or link of your preference).

In [ ]:
# Quiz 1

## Quiz 1 — The Hexagon Chaos Game

Write a function that does the following:

1. Create a regular hexagon and pick a random point inside it
2. Pick two adjacent vertices at random and form a triangle with your point
3. Compute the centroid of that triangle — this becomes your new point
4. Repeat 10,000 times, saving each centroid
5. Plot all 10,000 points as a scatter plot

What pattern emerges? Describe it in a markdown cell below your plot.

In [ ]:
# silent installation
# Reuse this code to install unavailable modules/programs
import subprocess, sys

def install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

install("cartopy")
install("geopandas")
install("rasterio")
install("tqdm")
install("gdown")

In [ ]:
# Relevant imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection

# geo
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling
from pathlib import Path

# mapping
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# utils
from tqdm import tqdm

# add any imports you need below

In [ ]:
# Solve quiz one here or in subsequent cells. You can use a single or multiple cells

In [ ]:
def create_hexagon(radius=1):
#first creates a function called create_hexagon with single parameter radius=1

    angles = np.linspace(0, 2*np.pi, 6, endpoint=False)
    #uses function linspace from numpy to create 6 equally spaced angles between 0 and 2pi, endpoint = False means not inclusive of 2pi

    x = radius * np.cos(angles)
    y = radius * np.sin(angles)
    #converts the angles from polar to cartesian coordinates using rcos(theta) and rsin(theta)

    vertices = np.column_stack((x, y))
    #creates stack of pairs of (x,y) coordinates

    return vertices
    #returns stack

hexagon = create_hexagon()
#code to call the function create_hexagont

Unfamiliar with how shapes were made in google Colab, so I had to ask ChatGPT the syntax and format of Python code for making a hexagon. The function part of creating the hexagon comes from ChatGPT and I attempt to explain the code through comments

In [ ]:
x_min = -1
x_max = 1
y_min = -1
y_max = 1
#creates bound for a square to randomly generate a coordinate from

x_rand = np.random.uniform(x_min, x_max)
y_rand = np.random.uniform(y_min, y_max)
#randomly generated value inside the restraints of the square

from matplotlib.path import Path

hex_path = Path(hexagon)
#creates geometric shape from hexagon stack / 2d array that bounds the area of the hexagon
point = np.array([x_rand, y_rand])
#creates a point for the random point

while not hex_path.contains_point(point):
  x_rand = np.random.uniform(x_min, x_max)
  y_rand = np.random.uniform(y_min, y_max)
  point = np.array([x_rand, y_rand])
#while loop that runs until Hexagon contains the point coordinate; if not, then randomly choose another point
#Once the point is chosen, I print the point, not necessary in code though

print(x_rand, " ", y_rand)

The code above has some borrowed syntax from google as well as ChatGPT, but the idea I focused on here is how to randomly generate the first point. I thought about making some sort of easily adjustable coordinate restraint to the hexagon such as a rectangle of square and to continue to randomly generate coordinates in that region until it fit inside the Hexagon. That would get me the first initial "random" point for calculations.

In [ ]:
points = []
#initializes an array points to store all centroid points

current_point = np.array([x_rand, y_rand])
#First initial point that was randomly selected inside the Hexgon

for i in range(10000):
#loop 10000 times
  i = np.random.randint(0, 6)
  #chooses first vertex index, not inclusive of 6
  neighbor = np.random.choice([ (i-1)%6, (i+1)%6 ])
  #chooses random adjacent vertex index left or right
  centroid = (current_point + hexagon[i] + hexagon[neighbor]) / 3
  #calculates centroid
  points.append(centroid)
  #adds to points array
  current_point = centroid
  #turns the centroid into new starting point

points = np.array(points)
# Convert to numpy array after all points are collected

plt.figure(figsize=(6,6))
plt.scatter(points[:,0], points[:,1], s=0.5)
#plots the x,y coordinates given the columns of the array points and scales its size down
plt.gca().set_aspect('equal')
plt.title("Hexagon Chaos Game")
plt.show()
#code for creating the scatterplot graph

Looks like one of those fractals you often see, like Sierpinski's triangle or Mandelbrot fractal, where the same is made up of the same shape when you zoom in and in and in. Honestly, this was not what I expected as I figured as I didn't expect some parts of the hexagon to be completely unmarked and empty. This scatterplot makes me think of one of those neverending pi circles where you can draw curves around a center over and over again and none of them would overlap.

## Quiz 2 — Flood Hazard & US Power Infrastructure

The goal of this quiz is to identify which high-voltage power infrastructure
is most exposed to riverine flooding using the WRI Aqueduct flood model.

### Data

**Flood raster — WRI Aqueduct**  
Global riverine flood hazard map at 100-year return period (~1km resolution).  
Source: https://www.wri.org/aqueduct

**Transmission lines — Homeland Infrastructure Foundation-Level Data (HIFLD)**  
National dataset of electric transmission lines maintained by the US government.
Filtered to lines operating at 345 kV and above.

**Substations — OpenStreetMap**  
Substation locations extracted from OSM and filtered to high-voltage substations (≥345 kV).

---

Together these three layers let us ask: *which critical power infrastructure
sits in a floodplain, and how deep could the water get in a 1-in-100 year event?*

In [ ]:
import os
# WRI Aqueduct S3 base URL
BASE_URL = "https://wri-projects.s3.amazonaws.com/AqueductFloodTool/download/v2"

FLOOD_TYPES = {
    "inunriver": "Riverine (fluvial) flooding",
    "inuncoast": "Coastal flooding"
}

# Using the naming convention from the script above, construct the filename
# for the historical riverine flood map at 100-year return period.
# Then use wget or curl to download it into a folder called data/aqueduct/
# Confirm the download with ls -lh

FILENAME = "inunriver_historical_000000000WATCH_1980_rp00100.tif"
#creates file name that needs to correspond with the s3 formatting to download data

os.makedirs("data/aqueduct", exist_ok=True)
#checks to see if the folder exists

!wget -O data/aqueduct/$FILENAME $BASE_URL/$FILENAME
#download with wget

!ls -lh data/aqueduct
#confirms the download was successful

After running this code, data was successfully taken from the s3 bucket and saved to the file name and folder data/aqueduct.

In [ ]:
# Now let's inspect the file we just downloaded
# Open the raster with rasterio and print the following:
# - CRS
# - Bounds
# - Resolution
# - Nodata value
from pathlib import Path
import rasterio

aqueduct_path = Path("data/aqueduct")


tifs = list(aqueduct_path.glob("*inunriver*rp00100*.tif"))
#finds all .tif files in data/aqueduct whose name contains inunriver and rp00100 and places it in a list

tif_file = aqueduct_path / FILENAME


with rasterio.open(tif_file) as src:
    print("\nRaster Metadata:")
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("NoData value:", src.nodata)
#opens the raster with rasterio and prints the desired values


Has looked into the file and printed the metadata of CRS, Bounds, Resolution, and NoData value.

1. What is the CRS of the file? Is it geographic or projected? EPSG 4326, geographic coordinates.
2. What does a resolution of 0.00833 degrees mean in kilometers? Approximately 1 km, so each raster cell is 1km by 1km area
3. Why would this CRS be a problem if you wanted to calculate flood area in km²? This CRS uses degrees for each pixel, not km, so pixel area will vary with latitude. This in turn will result in pixels near the poles to cover less area (km^2) than pixels near the equator, leading to potential overestimates of flood area near poles

In [ ]:
# Download hv_substations.parquet and hv_lines.parquet from the shared folder:
# https://drive.google.com/drive/folders/1kCa3dDEiAJzfMpux_-Wyu9FO3Ub5ukFL?usp=sharing
# Save both files into data/infrastructure/

import gdown

DATA_DIR = Path("data/infrastructure")
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "hv_substations.parquet": "1r1JfzEkXnZtVA2-gqCKOY79r-FF6pTUF",
    "hv_lines.parquet":       "16bYruT6xF59coDhOgewYnK3KMDJpu8K5",
}

for fname, fid in FILES.items():
    out = DATA_DIR / fname
    if not out.exists():
        gdown.download(f"https://drive.google.com/uc?id={fid}", str(out), quiet=False)
    else:
        print(f"already exists: {fname}")

# load and inspect
subs = gpd.read_parquet(DATA_DIR / "hv_substations.parquet")
lines = gpd.read_parquet(DATA_DIR / "hv_lines.parquet")

# your code here
# inspect the following for both subs and lines:
# - CRS
# - shape
# - column names
# - bounding coordinates (hint: .total_bounds)
# - first few rows (.head())

Answer the following:
1. What CRS are the files in?
2. Do subs and lines share the same CRS as the raster (EPSG:4326)?
3. How many substations and lines are in the dataset?
4. What columns look useful for the analysis?


In [ ]:
# Before sampling, make sure subs and lines are in EPSG:4326 to match the raster
# For lines, you need to sample along nodes — convert lines to points first
# hint: use .interpolate() or extract vertices from the geometry

# 1. reproject if necessary
# 2. convert lines to nodes (points along the line)
# 3. use sample_raster_fast() to sample flood depth at substations and nodes

def sample_raster_fast(gdf, raster_path, nodata, col_name, batch_size=50000):
    coords = [(g.x, g.y) for g in gdf.geometry]
    vals = []
    with rasterio.open(raster_path) as src:
        for i in tqdm(range(0, len(coords), batch_size), desc=f"Sampling {col_name}"):
            batch = coords[i:i+batch_size]
            vals.extend([v[0] for v in src.sample(batch)])
    vals = np.array(vals, dtype=float)
    vals[vals == nodata] = np.nan
    vals = np.clip(vals, 0, None)
    out = gdf.copy()
    out[col_name] = vals
    return out

with rasterio.open(aq_path) as src:
    aq_nodata = src.nodata

# your code here
# once sampled, compute the following for substations and lines:
# - mean, median, std, min, max flood depth
# - 25th, 50th, 75th, 95th percentile
# - how many substations have flood depth > 0?
# - filter hotspots at the 95th percentile threshold
# - print flood depth range, number of hotspots, and threshold values

Answer the following:
1. What is the difference between mean and median flood depth?
   Which is more appropriate here and why?
2. Why do we use the 95th percentile as the hotspot threshold
   rather than any depth > 0?
3. What does it mean physically that X substations are above the threshold?
4. Look at the standard deviation — what does a high std tell you
   about the flood depth distribution?


In [ ]:
# Plot the flood hotspots
# 1. plot transmission lines colored by voltage class
# 2. plot all substations as small gray dots
# 3. overlay hotspot substations as red squares
# 4. add a title and legend
# hint: geopandas has a built in .plot() method

fig, ax = plt.subplots(figsize=(8, 8))

# your code here

plt.show()